<a href="https://colab.research.google.com/github/SehbazSingh/Language-Model-Using-RNN/blob/main/Language_model_using_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building Vocab

In [27]:
text = '''
hi hello
how are you i am fine
what is ai ai is artificial intelligence
'''
words = text.split()
vocab = ["<UNK>"] + sorted(set(words))

word_to_index = {word: i for i, word in enumerate(vocab)}
index_to_word = {i: word for word, i in word_to_index.items()}


# Encode and Decode Functions

In [28]:
def encode(text):
    words = text.split()
    return [word_to_index.get(word, word_to_index["<UNK>"]) for word in words]

print(encode(text))


def decode(indices):
    decoded_words = []
    for index in indices:
        decoded_words.append(index_to_word.get(index, "<UNK>"))
    return ' '.join(decoded_words)
    print(decode([2, 1, 2]))

[7, 6, 8, 3, 13, 9, 2, 5, 12, 11, 1, 1, 11, 4, 10]


# Next Token Prediction

In [29]:
encoded = [2,1,2,1]

seq_length = 2
X = []
Y = []

for i in range(len(encoded) - seq_length):
    x = encoded[i:i+seq_length]
    y = encoded[i+1:i+seq_length+1]

    X.append(x)
    Y.append(y)

print(X)
print(Y)

[[2, 1], [1, 2]]
[[1, 2], [2, 1]]


# Building Neural Network

In [30]:
import torch
import torch.nn as nn

embed = nn.Embedding(len(vocab), 4)

rnn = nn.RNN(4, 8, batch_first=True)

fc = nn.Linear(8, 3)

X_tensor = torch.tensor(X)
emb = embed(X_tensor)
out,_ = rnn(emb)
output = fc(out)

print(output.shape)
print(output)

torch.Size([2, 2, 3])
tensor([[[ 0.1886,  0.2108, -0.1740],
         [ 0.3454,  0.2781, -0.4637]],

        [[ 0.3959,  0.2815, -0.3539],
         [ 0.2470,  0.2885, -0.2434]]], grad_fn=<AddBackward0>)


# Connecting Everything

In [31]:
Y_tensor = torch.tensor(Y) # converting Y to tensor

# reshaping the output
output = output.view(-1, 3)
Y_tensor = Y_tensor.view(-1)

# Loss Function
loss_fn = nn.CrossEntropyLoss()
loss = loss_fn(output, Y_tensor)

print(loss)





tensor(1.2998, grad_fn=<NllLossBackward0>)


# Training Loop

In [37]:
optimizer = torch.optim.Adam(
    list(embed.parameters()) +
    list(rnn.parameters()) +
    list(fc.parameters()),
    lr=0.01
)

for epoch in range(100):

    X_tensor = torch.tensor(X)
    Y_tensor = torch.tensor(Y)

    emb = embed(X_tensor)
    out, _ = rnn(emb)
    output = fc(out)

    output = output.view(-1, 3)
    Y_tensor = Y_tensor.view(-1)

    loss = loss_fn(output, Y_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(epoch, loss.item())

0 0.013118576258420944
10 0.0035450132563710213
20 0.0013423331547528505
30 0.0007406661752611399
40 0.0005044240388087928
50 0.00038628169568255544
60 0.0003156753955408931
70 0.0002677674638107419
80 0.00023225205950438976
90 0.0002044526336248964


# Making the model talk

In [34]:
input_seq = encode("hi") #Encode input
input_tensor = torch.tensor([input_seq]) #Converting to tensor

# Forward pass
emb = embed(input_tensor)
out, _ = rnn(emb)
output = fc(out)

last_output = output[0, -1] # Take LAST word prediction

predicted_index = torch.argmax(last_output).item() # Geting predicted word

print(index_to_word[predicted_index] )# Decode




am


# Predicting a sentence

In [38]:
words = ["what is ai"]   # starting input
for _ in range(5):
  # 1. Take last 2 words
  input_seq = encode(" ".join(words[-2:]))

  # 2. Convert to tensor
  input_tensor = torch.tensor([input_seq])

  # 3. Forward pass
  emb = embed(input_tensor)
  out, _ = rnn(emb)
  output = fc(out)

  # 4. Take last prediction
  last_output = output[0, -1]

  # 5. Get predicted word
  predicted_index = torch.argmax(last_output).item()

  # 6. Convert to word
  next_word = index_to_word[predicted_index]

  # 7. Add to sentence
  words.append(next_word)

print(" ".join(words))


what is ai am ai am ai am
